# 02 — Klasik Baseline: blob detection + Hungarian linking

Detection = **Otsu eşiği → çekirdek-boyutlu local-max → bağlı bileşen ağırlık merkezi**
(`center_of_mass`). Linking = **Hungarian, 8 µm kapılı**. Sıfır ek bağımlılık.

> **Not:** Bu hat, Erdem'in `erd_exp/eda_detailed` baseline'ından uyarlandı (yerelde
> ~0.78 micro edge-Jaccard, LB ~0.7). Bizim önceki `top-N + adaptif hedef` yaklaşımımız
> tek çekirdeği birden çok kez tespit edip (kare-içi komşu ~5 µm) linking'i kırıyordu;
> `center_of_mass` bunu tek merkeze indirgiyor. Ayrıntı: `docs/EDA_BULGULARI.md`.

> ### ⚠️ SUBMIT KURALLARI
> - **Internet KAPALI** (Settings → Internet = Off). `pip install` YOK.
> - **`erdeemt/cell-tracking-libs` dataset'i EKLİ** — `zarr` imajda yok.
> - **Code competition:** notebook gizli test setiyle yeniden çalışır → test isimleri
>   **hardcode edilmez**, `test/*.zarr` dinamik gezilir.
> - Çıktı: **`/kaggle/working/submission.csv`**.

### Metriğin şekli (neden recall'a odaklanıyoruz)
GT seyrek → bir tahmin kenarının **iki ucu da** GT node'a eşleşmezse kenar **yok sayılır**
(FP≈0). Yani `adjusted_jaccard ≈ (GT kenar recall) × ceza`. Ceza zayıf (`[0.9, 1.1]`),
fazla-tahmin neredeyse bedava. **Oyun: her GT kenarının iki ucuna tespit koy + bağla.**

## 0 — Kurulum (zarr utility dataset'ten)

In [ ]:
# zarr bu imajda YOK -> erdeemt/cell-tracking-libs dataset'inden sys.path ile.
# append! insert(0) DEGIL: pylibs'te numpy 2.5.1 var, ortamin 2.0.2'sini golgelememeli
# (scipy/skimage 2.0.2'ye karsi derlenmis).
import sys, os
from pathlib import Path
from collections import Counter

LIBS = Path('/kaggle/input/datasets/erdeemt/cell-tracking-libs/pylibs')
if not LIBS.exists():
    hits = [Path(r) for r, d, f in os.walk('/kaggle/input') if os.path.basename(r) == 'pylibs']
    assert hits, 'cell-tracking-libs dataset i notebook a ekli degil!'
    LIBS = hits[0]
sys.path.append(str(LIBS))

import time, warnings
import numpy as np
import pandas as pd
import zarr
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from skimage.filters import threshold_otsu
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
warnings.filterwarnings('ignore')

print('zarr ', zarr.__version__, '| numpy', np.__version__)
assert np.__version__.startswith('2.0'), f'numpy golgelendi: {np.__version__} — insert(0) mi kullandin?'
print('hazir')

## 1 — Yapılandırma & veri kökü (isim hardcode YOK)

In [ ]:
SCALE_ZYX = (1.625, 0.40625, 0.40625)   # um/piksel (Z,Y,X) — attrs'ten dogrulandi
S = np.array(SCALE_ZYX, dtype=np.float32)

LINK_MAX_UM = 8.0      # linking arama yaricapi (EDA: hareket 99p ~8 um)
MATCH_UM    = 7.0      # metrik eslesme toleransi

# --- Detection ---
# GUVENLI DEGERLER (Erdem'in 0.749 konfigu). Onceki thr×0.7 + FOOT (3,9,9) tuning'im
# 4 placeholder dataset'e OVERFIT etti + T_pred'i sisirdi -> LB'de dustu (0.739).
SIGMA      = (1, 2, 2)      # Gauss yumusatma (voxel; Z ekseni ince)
FOOT       = (3, 11, 11)    # local-max / peak_local_max footprint ~ cekirdek boyutu
THR_FACTOR = 1.0           # Otsu carpani (1.0 = ham Otsu). Esigi DUSURME -> T_pred sisirir.

# --- Watershed: DENENDI, ZARARLI cikti -> KAPALI ---
# Mesafe-donusumu (EDT) marker'lari bu veri icin YANLIS: Otsu temas eden cekirdekleri tek
# birlesik foreground'a yapistiriyor, EDT tepeleri cekirdek-basina-bir DEGIL (en buyuk ic-tegetin
# merkezi) -> dense dokuda UNDER-segment. Sonuc: 6bba_05db0fb1 pred/kare 662->166, recall 0.88->0.35;
# 44b6_0b24845f recall 0.33->0.00. Ayrica 3-4x yavas (timeout riski). Yogunluk local-max'i (asagida
# WATERSHED=False yolu, Erdem'in hatti) her cekirdegin parlak merkezini yakaliyor -> cok daha iyi.
# TEKRAR DENERSEK: marker'lar YOGUNLUK tepesinden gelmeli (EDT'den degil).
WATERSHED   = False
MIN_DIST_UM = 4.0

# --- Bolunme (division / mitosis) — Erdem'in link_and_divide mantigi ---
DIV_ON     = True
DIV_MAX_UM = 6.0           # ebeveyn-kiz maks mesafe (EDA D07 medyan ~6um; linking gate'ten DAR)
DIV_SIB_UM = 13.0          # iki kiz arasi maks mesafe (EDA D07 kiz-kiz 95p ~13um; FP filtresi)

# --- Yerel metrikte ceza (penalty) icin T_true tahmini ---
# Kaggle: adjusted = raw_J × max(0, 1 − 0.1·(T_pred − T_true)/T_true). T_true SUNUCUDA,
# bize verilmiyor. Yerel proxy: her dataset icin REFERANS blob sayimi (Otsu+FOOT11,
# watershed'siz) — bizim tuning knob'larindan BAGIMSIZ. T_pred bunu asarsa ceza gorunur.
TTRUE_REF_FRAMES = 6       # T_true tahmini icin ornekelenecek kare sayisi

MAX_TEST = None
OUT_CSV = Path('/kaggle/working/submission.csv')
INPUT = Path('/kaggle/input')

# Dogrulanmis yarisma yolu (auto-detect basarisiz olursa fallback)
COMP_DIR = INPUT / 'competitions' / 'biohub-cell-tracking-during-development'

def find_root():
    """train/ + test/ iceren yarisma dizinini bul (mount yolu degisebilir)."""
    if (COMP_DIR / 'train').is_dir() and (COMP_DIR / 'test').is_dir():
        return COMP_DIR
    st = [(INPUT, 0)]
    while st:
        b, d = st.pop()
        try:
            if (b / 'train').is_dir() and (b / 'test').is_dir():
                return b
        except Exception:
            pass
        if d < 5:
            try:
                for c in sorted(b.iterdir()):
                    if c.is_dir() and not c.name.endswith(('.zarr', '.geff')):
                        st.append((c, d + 1))
            except Exception:
                pass
    listing = []
    for r, dch, f in os.walk(INPUT):
        if r.count(os.sep) - str(INPUT).count(os.sep) <= 1:
            listing.append(r)
        else:
            dch[:] = []
    raise RuntimeError(
        'yarisma koku bulunamadi (train/ + test/ yok).\n'
        '>> COZUM: sag panel -> Add Input -> yarismayi (biohub-cell-tracking...) EKLE.\n'
        '/kaggle/input agaci (ilk 2 seviye):\n  ' + '\n  '.join(sorted(listing)))

ROOT = find_root(); TRAIN = ROOT / 'train'; TEST = ROOT / 'test'
test_names = sorted(p.stem for p in TEST.glob('*.zarr'))
print('ROOT:', ROOT)
print(f'train={len(list(TRAIN.glob("*.zarr")))} | test={len(test_names)}')
print('test:', test_names[:10], '...' if len(test_names) > 10 else '')

# DEV mi GERCEK RERUN mu? Placeholder test = train'den kopya -> GT'leri train'de.
DEV = len(test_names) > 0 and (TRAIN / (test_names[0] + '.geff')).exists()
print('\nMOD:', 'DEV (placeholder test, GT var)' if DEV else 'GERCEK RERUN (gizli test, GT yok)')

## 2 — Okuyucular

In [ ]:
def open_image(zpath):
    """OME-Zarr goruntu dizisini ac -> (T,Z,Y,X)."""
    n = zarr.open(str(zpath), mode='r')
    a = dict(n.attrs)
    ms = a.get('multiscales') or (a['ome'].get('multiscales') if isinstance(a.get('ome'), dict) else None)
    if ms:
        return n[ms[0]['datasets'][0]['path']]
    return n['0'] if '0' in list(n.keys()) else n

def load_geff(gp):
    """GEFF -> (nodes_df[id,t,z,y,x], edges (E,2)). Duz zarr; geff/tracksdata gerekmiyor."""
    g = zarr.open(str(gp), mode='r')
    ids = np.asarray(g['nodes/ids'])
    d = {'id': ids}
    for k in ('t', 'z', 'y', 'x'):
        d[k] = np.asarray(g[f'nodes/props/{k}/values'])
    return pd.DataFrame(d), np.asarray(g['edges/ids'])

print('ok')

## 3 — Detection: Otsu → yoğunluk local-max → ağırlık merkezi

Gauss → Otsu eşiği → çekirdek-boyutlu local-max → bağlı bileşen ağırlık merkezi. Her
çekirdeğin **parlak merkezi** bir tespit — dense dokuda bile bir-çekirdek-bir-marker verir.

> **Watershed denendi, ZARARLI çıktı → `WATERSHED=False`.** Mesafe-dönüşümü (EDT) marker'ları
> bu veri için yanlış: Otsu temas eden çekirdekleri tek foreground kütlesine yapıştırıyor,
> EDT tepeleri çekirdek-başına-bir değil → dense dokuda **under-segment** (6bba_05db0fb1
> recall 0.88→0.35). Yoğunluk local-max'ı bunu yaşamıyor. Tekrar denenirse marker'lar
> **yoğunluk tepesinden** gelmeli (EDT'den değil). Kod parametreli kaldı ama devre dışı.

In [ ]:
def _centers_plain(sm, thr):
    # Referans (watershed'siz): local-max peak bilesenlerinin agirlik merkezi.
    mx = ndi.maximum_filter(sm, size=FOOT)
    peaks = (sm == mx) & (sm > thr)
    lbl, n = ndi.label(peaks)
    if n == 0:
        return np.zeros((0, 3), np.float32)
    return np.asarray(ndi.center_of_mass(sm, lbl, np.arange(1, n + 1)), np.float32)


def detect_frame(v):
    # (Z,Y,X) -> (N,3) voxel merkez. WATERSHED=True ise bitisik cekirdekleri ayirir.
    sm = ndi.gaussian_filter(v.astype(np.float32), sigma=SIGMA)
    thr = threshold_otsu(sm) * THR_FACTOR
    if not WATERSHED:
        return _centers_plain(sm, thr)

    fg = sm > thr
    if not fg.any():
        return np.zeros((0, 3), np.float32)
    # anizotropi-farkinda mesafe donusumu (um): temas eden cekirdeklerin merkezleri ayri tepe verir
    dist = ndi.distance_transform_edt(fg, sampling=SCALE_ZYX)
    # marker'lar = mesafe tepe noktalari (yogunluk tepesi degil -> yogunluk-dip'i olmayan
    # temas eden cekirdekleri de ayirir). min_distance um -> voxel (XY ekseninden).
    min_dist_vox = max(1, int(round(MIN_DIST_UM / SCALE_ZYX[1])))
    coords = peak_local_max(dist, min_distance=min_dist_vox, labels=fg,
                            footprint=np.ones(FOOT), exclude_border=False)
    if len(coords) == 0:
        return _centers_plain(sm, thr)
    markers = np.zeros(fg.shape, np.int32)
    markers[tuple(coords.T)] = np.arange(1, len(coords) + 1)
    lbl = watershed(-dist, markers, mask=fg)       # foreground'u marker basina bir bolgeye ayir
    n = int(lbl.max())
    if n == 0:
        return np.zeros((0, 3), np.float32)
    # merkez = bolgenin yogunluk-agirlikli merkezi (cekirdek ortasina yakin)
    return np.asarray(ndi.center_of_mass(sm, lbl, np.arange(1, n + 1)), np.float32)


def estimate_true_count(arr, frames=TTRUE_REF_FRAMES):
    # T_true proxy: REFERANS detector (Otsu, FOOT11, watershed'siz) blob sayimi × T.
    # Bizim tuning knob'larindan bagimsiz -> T_pred'in bunu asmasi 'fazla-tahmin' isaretidir.
    T = arr.shape[0]
    ts = np.linspace(0, T - 1, min(frames, T)).astype(int)
    cnts = []
    for t in ts:
        sm = ndi.gaussian_filter(np.asarray(arr[int(t)]).astype(np.float32), sigma=SIGMA)
        cnts.append(len(_centers_plain(sm, threshold_otsu(sm))))
    return int(np.median(cnts) * T)


# hiz + yogunluk kontrolu (T'yi HARDCODE ETME)
_arr = open_image(TEST / (test_names[0] + '.zarr'))
_T = _arr.shape[0]
t0 = time.time(); _c = detect_frame(np.asarray(_arr[_T // 2])); dt = time.time() - t0
print(f'shape {_arr.shape} | WATERSHED={WATERSHED} | 1 kare {dt:.2f}s | cekirdek {len(_c)}')
print(f'tahmini 1 dataset ({_T} kare): {dt*_T:.0f}s | {len(test_names)} test: {dt*_T*len(test_names)/60:.1f} dk')

## 4 — Linking: Hungarian (8 µm) + **bölünme (mitosis)**

**1. tur:** optimal 1-1 eşleştirme; 8 µm üstü yasak. Eşleşmeyen = beliriş/kayboluş.
**2. tur (bölünme):** `t+1`'de eşleşmemiş bir node, `t`'de **zaten çocuğu olan** bir ebeveyne
≤6 µm ise → ikinci kız (ebeveyn 2-çıkışlı olur). İki kız ≤13 µm koşulu FP'yi eler
(EDA D07: ebeveyn-kız ~6 µm, kız-kız 95p ~13 µm). Skorun %10'u bölünmeden gelir.

Detection ayrıştırıldığı için (`cents` cache) aynı tespitlerle **bölünmeli/bölünmesiz**
karşılaştırma yapılabilir.

In [ ]:
def link_pairs(A, B):
    # 1-1 Hungarian eslestirme, 8 um kapili. Ayrica D matrisini doner (bolunme icin).
    if len(A) == 0 or len(B) == 0:
        return [], None
    D = cdist(A * S, B * S)                        # um mesafe
    cost = np.where(D <= LINK_MAX_UM, D, 1e6)      # kapi
    r, c = linear_sum_assignment(cost)
    return [(int(i), int(j)) for i, j in zip(r, c) if D[i, j] <= LINK_MAX_UM], D


def link_and_divide(A, B, div_on):
    # 1. tur: normal 1-1 linking. 2. tur (bolunme): t+1'de eslesmemis bir node, t'de ZATEN
    # cocugu olan bir ebeveyne <= DIV_MAX_UM ise -> ikinci kiz. Iki kiz <= DIV_SIB_UM kosulu FP eler.
    pairs, D = link_pairs(A, B)
    if not div_on or D is None or not pairs:
        return pairs, []
    child_of = {i: j for i, j in pairs}
    mA = np.array(sorted(child_of.keys()))
    matchedB = set(j for _, j in pairs)
    div = []
    for j in range(len(B)):
        if j in matchedB:
            continue
        cand = mA[D[mA, j] <= DIV_MAX_UM]          # yakin, zaten-cocuklu ebeveynler
        best_i, best_d = -1, DIV_MAX_UM + 1
        for i in cand:
            sib = float(np.linalg.norm((B[child_of[i]] - B[j]) * S))   # iki kiz arasi
            if D[i, j] < best_d and sib <= DIV_SIB_UM:
                best_d, best_i = D[i, j], int(i)
        if best_i >= 0:
            div.append((best_i, j))
    return pairs, div


def track_dataset(arr, div_on=None, cents=None):
    # cents verilirse yeniden detect etme (pahali kismi cache'le). div_on None -> DIV_ON.
    if div_on is None:
        div_on = DIV_ON
    if cents is None:
        cents = [detect_frame(np.asarray(arr[t])) for t in range(arr.shape[0])]
    nodes, edges, off, nid = [], [], [], 1
    for t, c in enumerate(cents):
        off.append(nid)
        for p in c:
            nodes.append((nid, t, int(round(p[0])), int(round(p[1])), int(round(p[2])))); nid += 1
    for t in range(len(cents) - 1):
        pairs, div = link_and_divide(cents[t], cents[t + 1], div_on)
        for i, j in pairs:
            edges.append((off[t] + i, off[t + 1] + j))
        for i, j in div:
            edges.append((off[t] + i, off[t + 1] + j))    # bolunme = ebeveynin 2. cikis kenari
    return nodes, edges

print('ok')

## 5 — Yerel metrik: **ceza-farkında** edge Jaccard + division Jaccard

Önceki büyük eksik: metriğimiz Kaggle'ın **fazla-tahmin cezasını** modellemiyordu, bu yüzden
tespit artırmayı hep ödül sanıp `T_pred` şişmesini görmüyordu (0.739'a düşüşün sebebi).

Artık gerçek formülü hesaplıyoruz:
```
adjusted_edge_J = raw_edge_J × max(0, 1 − 0.1·(T_pred − T_true)/T_true)
FINAL           = adjusted_edge_J + 0.1 × division_J
```
`T_true` sunucuda; yerel proxy olarak **referans blob sayımı** (Otsu+FOOT11, watershed'siz)
kullanıyoruz. `penalty < 1` çıkarsa `T_pred` referansı aşmış = fazla-tahmin uyarısı.

In [ ]:
def eval_vs_gt(nodes, edges, gt_ndf, gt_edges):
    # Node'lari 7 um ile GT'ye esle, sonra kenarlari + bolunmeleri karsilastir.
    # Dataset basina HAM sayaclar doner; micro (havuz) ust hucrede yapilir.
    pn = pd.DataFrame(nodes, columns=['node_id', 't', 'z', 'y', 'x'])
    gmap = {}                                      # pred_id -> gt_id (7 um eslesme)
    for t, g in gt_ndf.groupby('t'):
        p = pn[pn.t == int(t)]
        if len(p) == 0 or len(g) == 0:
            continue
        D = cdist(p[['z', 'y', 'x']].values * S, g[['z', 'y', 'x']].values * S)
        cost = np.where(D <= MATCH_UM, D, 1e6)
        r, c = linear_sum_assignment(cost)
        pid = p['node_id'].values; gid = g['id'].values
        for i, j in zip(r, c):
            if D[i, j] <= MATCH_UM:
                gmap[int(pid[i])] = int(gid[j])
    # --- Edge Jaccard (seyrek GT: iki ucu da eslesmeyen tahmin kenari yok sayilir) ---
    gtset = set((int(u), int(v)) for u, v in gt_edges)
    eTP = eFP = 0; cov = set()
    for u, v in edges:
        gu, gv = gmap.get(u), gmap.get(v)
        if gu is None or gv is None:
            continue
        if (gu, gv) in gtset:
            eTP += 1; cov.add((gu, gv))
        else:
            eFP += 1
    eFN = len(gtset) - len(cov)
    # --- Division Jaccard (yaklasik): GT'de 2-cikisli node -> eslesen tahminimiz de 2-cikisli mi ---
    gt_div = set(u for u, c in Counter(int(u) for u, _ in gt_edges).items() if c >= 2)
    pr_div = set(u for u, c in Counter(u for u, _ in edges).items() if c >= 2)
    gt2pr = {}
    for pid_, gid_ in gmap.items():
        gt2pr.setdefault(gid_, pid_)
    dTP = sum(1 for gd in gt_div if gt2pr.get(gd) in pr_div)
    dFP = sum(1 for pdv in pr_div if (pdv in gmap) and (gmap[pdv] not in gt_div))
    dFN = len(gt_div) - dTP
    return dict(eTP=eTP, eFP=eFP, eFN=eFN, dTP=dTP, dFP=dFP, dFN=dFN,
                node_recall=round(len(set(gmap.values())) / max(len(gt_ndf), 1), 4),
                pred_nodes=len(nodes), gt_div=len(gt_div))


def micro_final(rows, t_pred_total, t_true_total):
    # Resmi skor: adjusted = raw_edge_J × max(0, 1 − 0.1·(T_pred − T_true)/T_true);
    #             Final = adjusted + 0.1·div_J. Hepsi MICRO (havuzlanmis TP/FP/FN).
    eTP = sum(r['eTP'] for r in rows); eFP = sum(r['eFP'] for r in rows); eFN = sum(r['eFN'] for r in rows)
    dTP = sum(r['dTP'] for r in rows); dFP = sum(r['dFP'] for r in rows); dFN = sum(r['dFN'] for r in rows)
    raw_eJ = eTP / max(eTP + eFP + eFN, 1)
    penalty = max(0.0, 1.0 - 0.1 * (t_pred_total - t_true_total) / max(t_true_total, 1))
    adj_eJ = raw_eJ * penalty
    dJ = dTP / max(dTP + dFP + dFN, 1)
    return dict(raw_edge_J=round(raw_eJ, 4), penalty=round(penalty, 4), adj_edge_J=round(adj_eJ, 4),
                div_J=round(dJ, 4), FINAL=round(adj_eJ + 0.1 * dJ, 4),
                T_pred=t_pred_total, T_true_est=t_true_total,
                eTP=eTP, eFP=eFP, eFN=eFN, dTP=dTP, dFP=dFP, dFN=dFN)

print('ok')

### 5a — Sanity: GT → GT skoru **1.0** olmalı (yalnızca DEV)

In [ ]:
if DEV:
    g_ndf, g_edges = load_geff(TRAIN / (test_names[0] + '.geff'))
    gt_nodes = [(int(r.id), int(r.t), float(r.z), float(r.y), float(r.x)) for r in g_ndf.itertuples()]
    chk = eval_vs_gt(gt_nodes, [(int(u), int(v)) for u, v in g_edges], g_ndf, g_edges)
    raw = chk['eTP'] / max(chk['eTP'] + chk['eFP'] + chk['eFN'], 1)
    print('GT->GT: raw_edge_J =', round(raw, 4), '| eTP', chk['eTP'], 'eFP', chk['eFP'], 'eFN', chk['eFN'],
          '| dTP', chk['dTP'], 'gt_div', chk['gt_div'])
    assert raw > 0.999, 'SANITY FAIL — metrik implementasyonu hatali!'
    print('>> Metrik dogrulandi.')
else:
    print('GERCEK RERUN -> sanity atlandi (GT yok)')

## 6 — Baseline'ı değerlendir (test isimli 4 dataset, train GT ile)

Bu 4 film hem train hem test'te → GT'leri elimizde. Fit edilen parametre yok → **yanlı değil**,
gerçek skor tahmini. Resmi metrik **micro-average** (TP/FP/FN videolar arası havuzlanır).

In [ ]:
if DEV:
    # Detection PAHALI kisim -> bir kez yap, cache'le. Sonra bolunmesiz/bolunmeli + T_true.
    CACHE = {}
    for nm in test_names:
        gp = TRAIN / (nm + '.geff')
        if not gp.exists():
            continue
        t0 = time.time()
        arr = open_image(TEST / (nm + '.zarr'))
        cents = [detect_frame(np.asarray(arr[t])) for t in range(arr.shape[0])]
        CACHE[nm] = (cents, load_geff(gp), estimate_true_count(arr))
        print(f'  {nm}: detection {time.time()-t0:.0f}s | T_pred~{sum(len(c) for c in cents)} '
              f'| T_true_est~{CACHE[nm][2]}')

    T_true_total = sum(v[2] for v in CACHE.values())
    for div_on in [False, True]:
        rows = []
        T_pred_total = 0
        for nm, (cents, (gn, ge), _tt) in CACHE.items():
            nodes, edges = track_dataset(None, div_on=div_on, cents=cents)
            r = eval_vs_gt(nodes, edges, gn, ge); r['dataset'] = nm
            rows.append(r); T_pred_total += r['pred_nodes']
        m = micro_final(rows, T_pred_total, T_true_total)
        tag = 'BOLUNMELI ' if div_on else 'bolunmesiz'
        print(f'\n=== {tag} ===')
        for r in rows:
            raw = r['eTP'] / max(r['eTP'] + r['eFP'] + r['eFN'], 1)
            print(f"  {r['dataset']}: raw_edgeJ={raw:.3f} recall={r['node_recall']:.3f} "
                  f"eTP={r['eTP']} eFN={r['eFN']} | dTP={r['dTP']} dFP={r['dFP']} gt_div={r['gt_div']} "
                  f"| pred/kare={r['pred_nodes']//100}")
        print(f"  >> raw_edge_J={m['raw_edge_J']} × penalty={m['penalty']} = adj={m['adj_edge_J']} "
              f"| div_J={m['div_J']} | FINAL={m['FINAL']}")
        print(f"     T_pred={m['T_pred']} vs T_true_est={m['T_true_est']}  "
              f"(oran {m['T_pred']/max(m['T_true_est'],1):.2f}x)")
    print('\n(referans: onceki submit LB=0.749/0.739. penalty<1 ise T_pred sismis demektir.)')
else:
    print('GERCEK RERUN -> eval atlandi (sadece submission uretilir)')

## 7 — Gönderim: `test/` dinamik gez, `submission.csv` yaz

Satır satır yaz (gizli test büyük olabilir, RAM'de tutma). Bir dataset patlarsa yer tutucu
node koy, koşuyu öldürme — her test dataset'i submission'da yer almalı.

In [ ]:
import csv
gid = tot_n = tot_e = 0; failed = []; t_start = time.time()
with open(OUT_CSV, 'w', newline='') as fh:
    w = csv.writer(fh)
    w.writerow(['id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id'])
    for k, nm in enumerate(test_names, 1):
        t0 = time.time()
        try:
            nodes, edges = track_dataset(open_image(TEST / (nm + '.zarr')))
        except Exception as e:
            print(f'  [HATA] {nm}: {type(e).__name__}: {e}'); nodes, edges = [], []; failed.append(nm)
        for nid, t, z, y, x in nodes:
            w.writerow([gid, nm, 'node', nid, t, z, y, x, -1, -1]); gid += 1
        for u, v in edges:
            w.writerow([gid, nm, 'edge', -1, -1, -1, -1, -1, u, v]); gid += 1
        if not nodes:                                    # her dataset yer almali
            w.writerow([gid, nm, 'node', 1, 0, 0, 0, 0, -1, -1]); gid += 1
            print(f'  [uyari] {nm}: tespit yok -> yer tutucu')
        tot_n += len(nodes); tot_e += len(edges)
        print(f'[{k}/{len(test_names)}] {nm}: node={len(nodes)} edge={len(edges)} ({time.time()-t0:.0f}s)')
print(f'\nYAZILDI {OUT_CSV} | satir={gid:,} node={tot_n:,} edge={tot_e:,} | {(time.time()-t_start)/60:.1f} dk')
if failed:
    print('BASARISIZ (yer tutucu kondu):', failed)

## 8 — Şema doğrulama

In [ ]:
head = pd.read_csv(OUT_CSV, nrows=5)
ss = ROOT / 'sample_submission.csv'
if ss.exists():
    assert list(head.columns) == list(pd.read_csv(ss).columns), 'KOLON UYUSMAZLIGI!'
    print('kolonlar OK')
print(head.to_string(index=False))

if DEV:                                                  # agir kontrol sadece DEV
    sub = pd.read_csv(OUT_CSV)
    missing = set(test_names) - set(sub.dataset.unique())
    assert not missing, f'eksik dataset: {missing}'
    bad = 0
    for ds, g in sub.groupby('dataset'):
        nid = set(g[g.row_type == 'node'].node_id); e = g[g.row_type == 'edge']
        bad += int((~e.source_id.isin(nid)).sum() + (~e.target_id.isin(nid)).sum())
    assert bad == 0, f'gecersiz edge referansi: {bad}'
    print(f'row_type={dict(sub.row_type.value_counts())} | dataset={sub.dataset.nunique()} | edge-ref OK')
    print('>> Submission gecerli.')
else:
    print('GERCEK RERUN -> agir kontrol atlandi | satir:', sum(1 for _ in open(OUT_CSV)) - 1)
print('\nHatirlatma: Internet = OFF olmali, yoksa submit reddedilir.')

## 9 — Sonraki adımlar

Bu sürümde eklendi: **watershed instance ayrımı**, **bölünme detection**, **ceza-farkında
yerel metrik**. Değerlendirme (bölüm 6) kararı verir:

- **FINAL yükseldi + penalty ≈ 1** → watershed gerçek recall getirdi, T_pred şişmedi → **submit**.
- **penalty belirgin < 1** → watershed fazla parçaladı (over-segmentation); `MIN_DIST_UM` büyüt
  veya `WATERSHED=False` ile karşılaştır.
- **div_J düşük / dFP yüksek** → `DIV_MAX_UM`/`DIV_SIB_UM` sıkılaştır.

Sıradaki büyük levarlar:
- [ ] **Prefix-bazlı CV** (`train/` 199 dataset, 44b6/6bba fold) — 4 placeholder'a overfit'i
  önlemenin tek yolu. Watershed/parametreleri burada doğrula, LB'ye değil buna güven.
- [ ] **44b6_0b24845f** (recall ~0.33, param'a kör) — GT'yi görüntüye bindir; muhtemelen
  sönük/derin Z bölgesi. Belki `SIGMA` veya Z-özel eşik.
- [ ] **Watershed marker kaynağı** — mesafe-tepesi + yoğunluk-tepesi birleşimi (hibrit marker)
  hem yuvarlak hem düzensiz çekirdekleri yakalayabilir.
- [ ] **Runtime** — watershed + mesafe dönüşümü kareyi yavaşlatır; gizli test büyükse
  `sn/kare`'yi izle, gerekirse `float32`/downsample.